In [2]:
import os
import numpy as np
from datasets import load_dataset
from PIL import Image
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("TF version:", tf.__version__)
print("Keras version:", keras.__version__)

TF version: 2.19.0
Keras version: 3.12.1


In [7]:
pip install kagglehub


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 1.0 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 3.4 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [8]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("jjeevanprakash/nsfw-detection")

print("Path to dataset files:", path)

100%|██████████| 1.64G/1.64G [08:00<00:00, 3.65MB/s]

Extracting files...


Path to dataset files: /Users/palakadsul/.cache/kagglehub/datasets/jjeevanprakash/nsfw-detection/versions/1


In [9]:
import os

path = "/Users/palakadsul/.cache/kagglehub/datasets/jjeevanprakash/nsfw-detection/versions/1"

for root, dirs, files in os.walk(path):
    level = root.replace(path, "").count(os.sep)
    if level < 3:
        indent = " " * 2 * level
        print(f"{indent}{os.path.basename(root)}/")
        if level == 2:
            print(f"{indent}  files: {len(files)}")

1/
  out/
    test/
      files: 0
    train/
      files: 0
    val/
      files: 0


In [10]:
import os

path = "/Users/palakadsul/.cache/kagglehub/datasets/jjeevanprakash/nsfw-detection/versions/1"

for root, dirs, files in os.walk(path):
    level = root.replace(path, "").count(os.sep)
    if level < 4:
        indent = " " * 2 * level
        print(f"{indent}{os.path.basename(root)}/  ({len(files)} files)")

1/  (0 files)
  out/  (0 files)
    test/  (0 files)
      NSFW/  (3351 files)
      Neutral/  (2196 files)
    train/  (0 files)
      NSFW/  (10051 files)
      Neutral/  (6587 files)
    val/  (0 files)
      NSFW/  (3350 files)
      Neutral/  (2196 files)


In [11]:
import os
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

DATASET_PATH = "/Users/palakadsul/.cache/kagglehub/datasets/jjeevanprakash/nsfw-detection/versions/1/out"
IMG_SIZE = 224
BATCH_SIZE = 32

# Load train and test data directly from the existing folders
train_gen = keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    zoom_range=0.1,
    horizontal_flip=True
)
test_gen = keras.preprocessing.image.ImageDataGenerator(rescale=1./255)

train_data = train_gen.flow_from_directory(
    f"{DATASET_PATH}/train",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="binary"
)
test_data = test_gen.flow_from_directory(
    f"{DATASET_PATH}/test",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False
)

print("Classes:", train_data.class_indices)
print("Train samples:", train_data.samples)
print("Test samples:", test_data.samples)

Found 16638 images belonging to 2 classes.
Found 5547 images belonging to 2 classes.
Classes: {'NSFW': 0, 'Neutral': 1}
Train samples: 16638
Test samples: 5547


In [12]:
# Build EfficientNetB0 model
base = keras.applications.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
base.trainable = False

inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs)

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy", keras.metrics.AUC(name="auc")]
)

model.summary()

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,382,884 (16.72 MB)

 Trainable params: 330,753 (1.26 MB)

 Non-trainable params: 4,052,131 (15.46 MB)

In [14]:
print("Phase 1: Training head only (5 epochs)...")
history1 = model.fit(
    train_data,
    epochs=5,
    validation_data=test_data,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(patience=1, factor=0.5)
    ]
)
print("Phase 1 complete!")

Phase 1: Training head only (5 epochs)...
Epoch 1/5
520/520 ━━━━━━━━━━━━━━━━━━━━ 272s 517ms/step - accuracy: 0.5493 - auc: 0.5169 - loss: 0.7884 - val_accuracy: 0.6041 - val_auc: 0.5782 - val_loss: 0.6875 - learning_rate: 0.0010
Epoch 2/5
520/520 ━━━━━━━━━━━━━━━━━━━━ 404s 778ms/step - accuracy: 0.5878 - auc: 0.5373 - loss: 0.6824 - val_accuracy: 0.6041 - val_auc: 0.6741 - val_loss: 0.6757 - learning_rate: 0.0010
Epoch 3/5
520/520 ━━━━━━━━━━━━━━━━━━━━ 426s 819ms/step - accuracy: 0.5956 - auc: 0.5410 - loss: 0.6744 - val_accuracy: 0.6041 - val_auc: 0.6722 - val_loss: 0.6752 - learning_rate: 0.0010
Epoch 4/5
520/520 ━━━━━━━━━━━━━━━━━━━━ 362s 695ms/step - accuracy: 0.6019 - auc: 0.5514 - loss: 0.6707 - val_accuracy: 0.6041 - val_auc: 0.6827 - val_loss: 0.6737 - learning_rate: 0.0010
Epoch 5/5
520/520 ━━━━━━━━━━━━━━━━━━━━ 386s 743ms/step - accuracy: 0.6023 - auc: 0.5578 - loss: 0.6690 - val_accuracy: 0.6041 - val_auc: 0.6841 - val_loss: 0.6690 - learning_rate: 0.0010
Phase 1 complete!


In [15]:
print("Phase 2: Fine-tuning last 30 layers...")

# Unfreeze last 30 layers of EfficientNetB0
base_model = model.layers[1]  # EfficientNetB0 is layer index 1
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

print(f"Trainable layers: {sum(1 for l in model.layers if l.trainable)}")

# Recompile with much smaller learning rate
model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy", keras.metrics.AUC(name="auc")]
)

history2 = model.fit(
    train_data,
    epochs=10,
    validation_data=test_data,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.5)
    ]
)
print("Phase 2 complete!")

Phase 2: Fine-tuning last 30 layers...
Trainable layers: 7
Epoch 1/10
520/520 ━━━━━━━━━━━━━━━━━━━━ 326s 621ms/step - accuracy: 0.5904 - auc: 0.5250 - loss: 0.6804 - val_accuracy: 0.6041 - val_auc: 0.6751 - val_loss: 0.6697 - learning_rate: 1.0000e-05
Epoch 2/10
520/520 ━━━━━━━━━━━━━━━━━━━━ 345s 663ms/step - accuracy: 0.6090 - auc: 0.5703 - loss: 0.6638 - val_accuracy: 0.6165 - val_auc: 0.6462 - val_loss: 0.6690 - learning_rate: 1.0000e-05
Epoch 3/10
520/520 ━━━━━━━━━━━━━━━━━━━━ 355s 682ms/step - accuracy: 0.6167 - auc: 0.5941 - loss: 0.6550 - val_accuracy: 0.6376 - val_auc: 0.6517 - val_loss: 0.6668 - learning_rate: 1.0000e-05
Epoch 4/10
520/520 ━━━━━━━━━━━━━━━━━━━━ 376s 723ms/step - accuracy: 0.6247 - auc: 0.6103 - loss: 0.6483 - val_accuracy: 0.6330 - val_auc: 0.6820 - val_loss: 0.6585 - learning_rate: 1.0000e-05
Epoch 5/10
520/520 ━━━━━━━━━━━━━━━━━━━━ 355s 682ms/step - accuracy: 0.6311 - auc: 0.6343 - loss: 0.6393 - val_accuracy: 0.6467 - val_auc: 0.7025 - val_loss: 0.6582 - learnin

In [16]:
import os

# Save current model as backup first
model_save_path = os.path.expanduser("~/content_moderation/model/nsfw_model.h5")
model.save(model_save_path)
print(f"Backup saved: {model_save_path}")
size = os.path.getsize(model_save_path) / (1024*1024)
print(f"Size: {round(size, 1)} MB")

Backup saved: /Users/palakadsul/content_moderation/model/nsfw_model.h5
Size: 31.1 MB


In [17]:
# Reset data generators with class weights
import numpy as np

# Calculate class weights to fix imbalance
total = 16638
nsfw_count = 10051
neutral_count = 6587

weight_nsfw = total / (2 * nsfw_count)      # 0.827
weight_neutral = total / (2 * neutral_count) # 1.263

class_weights = {0: weight_nsfw, 1: weight_neutral}
print(f"Class weights: {class_weights}")

# Unfreeze more layers
base_model = model.layers[1]
base_model.trainable = True
for layer in base_model.layers[:-50]:
    layer.trainable = False

trainable_count = sum(1 for l in base_model.layers if l.trainable)
print(f"Trainable base layers: {trainable_count}")

# Recompile with slightly higher learning rate
model.compile(
    optimizer=keras.optimizers.Adam(3e-5),
    loss="binary_crossentropy",
    metrics=["accuracy", keras.metrics.AUC(name="auc")]
)

# Retrain
history3 = model.fit(
    train_data,
    epochs=10,
    validation_data=test_data,
    class_weight=class_weights,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.5)
    ]
)
print("Retraining complete!")

Class weights: {0: 0.8276788379265745, 1: 1.2629421587976317}
Trainable base layers: 30
Epoch 1/10
520/520 ━━━━━━━━━━━━━━━━━━━━ 262s 497ms/step - accuracy: 0.6227 - auc: 0.6493 - loss: 0.6539 - val_accuracy: 0.6654 - val_auc: 0.7858 - val_loss: 0.6104 - learning_rate: 3.0000e-05
Epoch 2/10
520/520 ━━━━━━━━━━━━━━━━━━━━ 293s 563ms/step - accuracy: 0.6198 - auc: 0.6675 - loss: 0.6421 - val_accuracy: 0.6331 - val_auc: 0.7885 - val_loss: 0.5966 - learning_rate: 3.0000e-05
Epoch 3/10
520/520 ━━━━━━━━━━━━━━━━━━━━ 1033s 2s/step - accuracy: 0.6213 - auc: 0.6731 - loss: 0.6369 - val_accuracy: 0.6454 - val_auc: 0.8016 - val_loss: 0.5791 - learning_rate: 3.0000e-05
Epoch 4/10
520/520 ━━━━━━━━━━━━━━━━━━━━ 297s 571ms/step - accuracy: 0.6169 - auc: 0.6783 - loss: 0.6328 - val_accuracy: 0.6656 - val_auc: 0.7985 - val_loss: 0.5871 - learning_rate: 3.0000e-05
Epoch 5/10
520/520 ━━━━━━━━━━━━━━━━━━━━ 303s 582ms/step - accuracy: 0.6195 - auc: 0.6822 - loss: 0.6320 - val_accuracy: 0.6454 - val_auc: 0.8121 -

In [18]:
model_save_path = os.path.expanduser("~/content_moderation/model/nsfw_model.h5")
model.save(model_save_path)
print(f"Improved model saved!")
size = os.path.getsize(model_save_path) / (1024*1024)
print(f"Size: {round(size, 1)} MB")

Improved model saved!
Size: 31.1 MB


In [19]:
import numpy as np
from tensorflow.keras.preprocessing import image as keras_image

# Test with a sample image from the dataset
test_img_path = "/Users/palakadsul/.cache/kagglehub/datasets/jjeevanprakash/nsfw-detection/versions/1/out/test/Neutral/1.jpg"

img = keras_image.load_img(test_img_path, target_size=(224, 224))
arr = keras_image.img_to_array(img) / 255.0
arr = np.expand_dims(arr, axis=0)

prob = float(model.predict(arr, verbose=0)[0][0])
print(f"Raw probability: {prob}")
print(f"Prediction: {'Neutral/Safe' if prob >= 0.5 else 'NSFW/Unsafe'}")

FileNotFoundError: [Errno 2] No such file or directory: '/Users/palakadsul/.cache/kagglehub/datasets/jjeevanprakash/nsfw-detection/versions/1/out/test/Neutral/1.jpg'

In [20]:
import os

neutral_path = "/Users/palakadsul/.cache/kagglehub/datasets/jjeevanprakash/nsfw-detection/versions/1/out/test/Neutral"
files = os.listdir(neutral_path)
print("First 5 files:", files[:5])

First 5 files: ['flower_0737.jpg', 'fruit_0819.jpg', 'dog_0097.jpg', 'motorbike_0491.jpg', 'motorbike_0485.jpg']


In [21]:
import numpy as np
from tensorflow.keras.preprocessing import image as keras_image

test_img_path = "/Users/palakadsul/.cache/kagglehub/datasets/jjeevanprakash/nsfw-detection/versions/1/out/test/Neutral/flower_0737.jpg"

img = keras_image.load_img(test_img_path, target_size=(224, 224))
arr = keras_image.img_to_array(img) / 255.0
arr = np.expand_dims(arr, axis=0)

prob = float(model.predict(arr, verbose=0)[0][0])
print(f"Raw probability: {prob}")
print(f"Prediction: {'Neutral/Safe' if prob >= 0.5 else 'NSFW/Unsafe'}")

# Test NSFW image too
nsfw_path = "/Users/palakadsul/.cache/kagglehub/datasets/jjeevanprakash/nsfw-detection/versions/1/out/test/NSFW"
nsfw_files = os.listdir(nsfw_path)
print(f"\nFirst NSFW file: {nsfw_files[0]}")

img2 = keras_image.load_img(f"{nsfw_path}/{nsfw_files[0]}", target_size=(224, 224))
arr2 = keras_image.img_to_array(img2) / 255.0
arr2 = np.expand_dims(arr2, axis=0)

prob2 = float(model.predict(arr2, verbose=0)[0][0])
print(f"Raw probability: {prob2}")
print(f"Prediction: {'Neutral/Safe' if prob2 >= 0.5 else 'NSFW/Unsafe'}")

Raw probability: 0.43950405716896057
Prediction: NSFW/Unsafe

First NSFW file: 6ugx2q5z8mv71_jpg.rf.1fa5f41c8b144730c1c7b38072e71214.jpg
Raw probability: 0.2155991643667221
Prediction: NSFW/Unsafe


In [1]:
# In Colab/Jupyter - run this once
import tensorflow as tf

model = tf.keras.models.load_model("nsfw_final_v2.keras")
model.save("nsfw_model.h5")  # save as H5 format
print("Saved!")

ValueError: File not found: filepath=nsfw_final_v2.keras. Please ensure the file is an accessible `.keras` zip file.

In [2]:
import os
print(os.path.exists("model/nsfw_model.h5"))

True
